In [ ]:
!conda install tensorflow

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [2]:
tf.__version__

'2.13.0'

In [4]:
data = pd.read_csv('cardio_train1.csv')
data.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [5]:
X = data.iloc[:, 1:-1]
y = data.iloc[:, -1]

In [6]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
X = mms.fit_transform(X)

In [7]:
np.shape(X)

(70000, 11)

In [8]:
y = pd.get_dummies(y)

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=33123)

In [32]:
_, _, _, y_test_r = train_test_split(X, data.iloc[:, -1], random_state=33123)

In [11]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(11, activation='relu', input_shape=(11,)),
    tf.keras.layers.Dense(30, activation='relu'),
    tf.keras.layers.Dense(60, activation='relu'),
    tf.keras.layers.Dense(80, activation='relu'),
    tf.keras.layers.Dense(40, activation='relu'),
    tf.keras.layers.Dense(25, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=tf.keras.losses.CategoricalCrossentropy(), metrics=[tf.keras.metrics.CategoricalAccuracy()])

In [12]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 11)                132       
                                                                 
 dense_1 (Dense)             (None, 30)                360       
                                                                 
 dense_2 (Dense)             (None, 60)                1860      
                                                                 
 dense_3 (Dense)             (None, 80)                4880      
                                                                 
 dense_4 (Dense)             (None, 40)                3240      
                                                                 
 dense_5 (Dense)             (None, 25)                1025      
                                                                 
 dense_6 (Dense)             (None, 2)                 5

In [15]:
model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test), batch_size=50)

Epoch 1/50
1050/1050 [==============================] - 3s 3ms/step - loss: 0.6145 - categorical_accuracy: 0.6593 - val_loss: 0.6172 - val_categorical_accuracy: 0.6571
Epoch 2/50
1050/1050 [==============================] - 3s 3ms/step - loss: 0.6132 - categorical_accuracy: 0.6603 - val_loss: 0.6183 - val_categorical_accuracy: 0.6537
Epoch 3/50
1050/1050 [==============================] - 3s 3ms/step - loss: 0.6118 - categorical_accuracy: 0.6640 - val_loss: 0.6133 - val_categorical_accuracy: 0.6644
Epoch 4/50
1050/1050 [==============================] - 3s 3ms/step - loss: 0.6096 - categorical_accuracy: 0.6648 - val_loss: 0.6199 - val_categorical_accuracy: 0.6466
Epoch 5/50
1050/1050 [==============================] - 3s 3ms/step - loss: 0.6055 - categorical_accuracy: 0.6710 - val_loss: 0.6102 - val_categorical_accuracy: 0.6591
Epoch 6/50
1050/1050 [==============================] - 3s 3ms/step - loss: 0.6002 - categorical_accuracy: 0.6772 - val_loss: 0.6184 - val_categorical_accuracy:

In [19]:
model.predict(X_test)[3][1] > 0.3

547/547 [==============================] - 1s 1ms/step


0.7908607

In [34]:
y_pred = model.predict(X_test)

547/547 [==============================] - 1s 1ms/step


In [35]:
y_pred_n1 = y_pred[:, 1]

In [36]:
y_pred_n1[y_pred_n1 > 0.40] = 1
y_pred_n1[y_pred_n1 <= 0.40] = 0

In [37]:
y_pred_n1

array([1., 1., 1., ..., 1., 1., 0.], dtype=float32)

In [38]:
from sklearn.metrics import classification_report
print(classification_report(y_true=y_test_r, y_pred=y_pred_n1))

              precision    recall  f1-score   support

           0       0.75      0.62      0.68      8652
           1       0.68      0.80      0.74      8848

    accuracy                           0.71     17500
   macro avg       0.72      0.71      0.71     17500
weighted avg       0.72      0.71      0.71     17500

